In [ ]:
import scanpy as sc
import scvelo as scv
adata = sc.read_h5ad(
    "../data/processed_conbine_data.h5ad"
)

print(adata)
print("layers:", adata.layers.keys())
print("raw:", adata.raw)
#print(adata.obs)
adata.obs["condition"].unique()

In [ ]:
SEED = 123

import random
import numpy as np
import torch
import scanpy as sc

# python
random.seed(SEED)

# numpy
np.random.seed(SEED)

# pytorch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# cuda deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# scanpy / scvelo
sc.settings.seed = SEED

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_neighbors=30, n_pcs=30)

In [ ]:
adata.obs['celltype.anno'].value_counts()

In [ ]:
import scvelo as scv
import scanpy as sc

scv.settings.verbosity = 3
scv.settings.set_figure_params("scvelo")

# 计算 velocity
scv.tl.velocity(adata, mode="dynamical")

# velocity graph
#scv.tl.velocity_graph(adata)

In [ ]:
""" scv.tl.recover_dynamics(adata)   # 这一步会产生 fit_likelihood
scv.tl.velocity(adata, mode="dynamical")
scv.tl.velocity_graph(adata)  """

In [ ]:
adata.obsm.keys()

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
adata.write("../data/scvelo_output_2.h5ad")

In [ ]:
adata.obs['root_cells'] = adata.obs['celltype.anno'] == 'Stem Cell Niche'

In [ ]:
import scvelo as scv
import scanpy as sc

# 1 重新 neighbors
sc.pp.neighbors(adata)

# 2 重新 moments
scv.pp.moments(adata)

# 3 重新 velocity graph
#scv.tl.velocity_graph(adata)

# 4 设置 root cell
adata.obs['root_cells'] = adata.obs['celltype.anno'] == 'Stem Cell Niche'

# 5 重新 latent time
scv.tl.latent_time(adata)

In [ ]:
scv.tl.latent_time(adata)

In [ ]:
""" scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="celltype.anno",
    legend_loc=None
) """
import scvelo as scv
import matplotlib.pyplot as plt
import os

# 强制白色背景
plt.style.use("default")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"

# 创建目录
os.makedirs("other_photo/scvelo", exist_ok=True)

fig, ax = plt.subplots(figsize=(6,6), facecolor="white")

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="celltype.anno",
    legend_loc=None,
    ax=ax,
    show=False
)

ax.set_title("ScVelo", fontsize=16)

fig.savefig(
    "other_photo/scvelo/scvelo_velocity_stream.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

In [ ]:
scv.tl.velocity_pseudotime(adata)

In [ ]:
%matplotlib inline

In [ ]:
""" ax = scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_pseudotime",
    cmap="plasma",
    colorbar=True,
    show=False
)

fig = ax.get_figure()
fig.savefig("velocity_pseudotime.png") """
import scvelo as scv
import matplotlib.pyplot as plt
import os

# 创建目录
os.makedirs("other_photo/scvelo", exist_ok=True)

ax = scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_pseudotime",
    cmap="plasma",
    colorbar=True,
    show=False
)

# 添加标题
ax.set_title("ScVelo", fontsize=16)

# 获取figure
fig = ax.get_figure()

# 保存图片
fig.savefig(
    "other_photo/scvelo/scvelo_velocity_pseudotime.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

In [ ]:
""" import scanpy as sc

sc.pl.umap(
    adata,
    color=["celltype.anno", "condition"],
    wspace=0.4
) """
import scanpy as sc
import matplotlib.pyplot as plt
import os

# 创建目录
os.makedirs("other_photo/scvelo", exist_ok=True)

# 画图
sc.pl.umap(
    adata,
    color=["celltype.anno", "condition"],
    wspace=0.4,
    show=False
)

# 获取figure
fig = plt.gcf()

# 添加总标题
fig.suptitle("ScVelo", fontsize=16)

# 保存
fig.savefig(
    "other_photo/scvelo/scvelo_condition.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

In [ ]:
""" scv.tl.latent_time(adata)

scv.pl.scatter(
    adata,
    color="latent_time",
    cmap="viridis"
) """
import scvelo as scv
import matplotlib.pyplot as plt
import os

# 计算 latent time
scv.tl.latent_time(adata)

# 创建目录
os.makedirs("other_photo/scvelo", exist_ok=True)

# 画图
ax = scv.pl.scatter(
    adata,
    basis="umap",
    color="latent_time",
    cmap="viridis",
    show=False
)

# 添加标题
ax.set_title("ScVelo", fontsize=16)

# 获取figure
fig = ax.get_figure()

# 保存图片
fig.savefig(
    "other_photo/scvelo/scvelo_latent_time.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import issparse

# spliced expression
X = adata.layers["spliced"]

if issparse(X):
    X = X.toarray()

# velocity
V = adata.layers["velocity"]

if issparse(V):
    V = V.toarray()

# 去掉 NaN
V = np.nan_to_num(V)

connectivities = adata.obsp["connectivities"]

scores = []

n_cells = X.shape[0]

for i in range(n_cells):

    vi = V[i].reshape(1, -1)

    neighbors = connectivities[i].indices

    sims = []

    for j in neighbors:

        delta = (X[j] - X[i]).reshape(1, -1)

        sim = cosine_similarity(vi, delta)[0][0]

        sims.append(sim)

    if len(sims) > 0:
        scores.append(np.mean(sims))

velocity_consistency = np.mean(scores)

print("Velocity Consistency:", velocity_consistency)

In [ ]:
import scvelo as scv

scv.tl.velocity_confidence(adata)

In [ ]:
""" scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_confidence",
    cmap="coolwarm"
) """
import scvelo as scv
import matplotlib.pyplot as plt
import os

# 如果还没计算 confidence，需要先算
scv.tl.velocity_confidence(adata)

# 创建目录
os.makedirs("other_photo/scvelo", exist_ok=True)

# 画图
ax = scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_confidence",
    cmap="coolwarm",
    show=False
)

# 添加标题
ax.set_title("ScVelo", fontsize=16)

# 获取figure
fig = ax.get_figure()

# 保存图片
fig.savefig(
    "other_photo/scvelo/scvelo_velocity_confidence.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

In [ ]:
print(adata.shape)
print(adata.layers["velocity"].shape)
print(adata.layers["spliced"].shape)
print(adata.layers["unspliced"].shape)

In [ ]:
adata.uns.keys()

In [ ]:
import numpy as np

adata.layers["velocity"] = np.array(adata.layers["velocity"])

In [ ]:
import scanpy as sc
import scvelo as scv
import numpy as np

# 1 确保 neighbors 存在
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30)

# 2 确保 velocity 是 numpy
adata.layers["velocity"] = np.array(adata.layers["velocity"])

# 3 重新计算 velocity graph



#scv.tl.velocity_graph(adata)

# 4 计算 confidence 和 consistency
scv.tl.velocity_confidence(adata)

In [ ]:
print("Mean velocity confidence:",
      adata.obs["velocity_confidence"].mean())

In [ ]:
print("velocity_confidence" in adata.obs)
print("velocity_consistency" in adata.obs)

In [ ]:
import scvelo as scv

scv.tl.velocity_confidence(adata)

In [ ]:
adata.obs.columns

In [ ]:
import matplotlib.pyplot as plt
import os

# 创建目录
os.makedirs("other_photo/scvelo", exist_ok=True)

# 画图
ax = scv.pl.scatter(
    adata,
    basis="umap",
    color="velocity_consistency",
    cmap="coolwarm",
    show=False
)

# 标题
ax.set_title("ScVelo", fontsize=16)

# 获取figure
fig = ax.get_figure()

# 保存
fig.savefig(
    "other_photo/scvelo/scvelo_velocity_consistency.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# expression matrix
X = adata.layers["spliced"]
if not isinstance(X, np.ndarray):
    X = X.toarray()

# velocity matrix
V = adata.layers["velocity"]
if not isinstance(V, np.ndarray):
    V = V.toarray()

# neighbor graph
connectivities = adata.obsp["connectivities"]

scores = []

for i in range(adata.n_obs):

    neighbors = connectivities[i].indices

    if len(neighbors) == 0:
        scores.append(0)
        continue

    vi = V[i].reshape(1, -1)

    sims = []

    for j in neighbors:
        delta = (X[j] - X[i]).reshape(1, -1)

        sim = cosine_similarity(vi, delta)[0][0]
        sims.append(sim)

    scores.append(np.mean(sims))

adata.obs["velocity_consistency"] = scores

In [ ]:
import scvelo as scv
import pandas as pd
import numpy as np

def velocity_benchmark(adata, model_name):

    results = {}

    # 1 velocity confidence
    if "velocity_confidence" not in adata.obs:
        scv.tl.velocity_confidence(adata)

    conf = adata.obs["velocity_confidence"]

    results["model"] = model_name
    results["confidence_mean"] = np.mean(conf)
    results["confidence_std"] = np.std(conf)

    # 2 velocity consistency
    if "velocity_consistency" not in adata.obs:
        scv.tl.velocity_confidence(adata)

    cons = adata.obs["velocity_consistency"]

    results["consistency_mean"] = np.mean(cons)
    results["consistency_std"] = np.std(cons)

    # 3 velocity pseudotime
    if "velocity_pseudotime" not in adata.obs:
        scv.tl.velocity_pseudotime(adata)

    pt = adata.obs["velocity_pseudotime"]

    results["pseudotime_mean"] = np.mean(pt)
    results["pseudotime_std"] = np.std(pt)

    # 4 latent time
    if "latent_time" not in adata.obs:
        scv.tl.latent_time(adata)

    lt = adata.obs["latent_time"]

    results["latent_time_mean"] = np.mean(lt)
    results["latent_time_std"] = np.std(lt)

    # 5 velocity magnitude
    v = adata.layers["velocity"]
    mag = np.linalg.norm(v, axis=1)

    results["velocity_mag_mean"] = np.mean(mag)
    results["velocity_mag_std"] = np.std(mag)

    return results

In [ ]:
results = []
results.append(velocity_benchmark(adata, "scvelo"))

df = pd.DataFrame(results)

print(df)

In [ ]:
import torch
print(torch.cuda.is_available())